# NFF-MTL on Steam — corrected pipeline, Colab driver (v0.3.1)

The notebook only drives the CLI. **Every command raises on failure, so "Run all" stops at the first error.**
Each command prints `RUN_DIR=...`; the notebook passes those exact folders on and never globs.

| Stage | Config | Reads test-year performance | Purpose |
| --- | --- | --- | --- |
| probe (`pilot`) | `configs/probe.yaml` | no | 300 batches, 2 000 validation users: seconds per batch, peak GPU memory |
| pilot | `configs/pilot.yaml` | no | full data, 2 epochs: real epoch and evaluation cost |
| tuning | `configs/default.yaml` | no | equal-budget search on 2021 → `final_selected.yaml` |
| final | `final_selected.yaml` | once per model identity | retrain with frozen settings, `evaluate` on 2022 |

Test-year *rows* are read for descriptive counts (data card) in every stage; test-year *scores* only in `final`.
The test registry next to the data cache is a guard against accidental repeated evaluation, not a tamper-proof lock.

**First run: "Run all" stops after the probe by design** (`PROCEED_TO_PILOT = False`), and again before tuning
(`PROCEED_TO_TUNE = False`). Change a switch only after reading the numbers printed above it.

In [ ]:
import json, subprocess, sys
from pathlib import Path

def run(*args):
    """Run a command, stream its output, raise CalledProcessError on a non-zero exit; return RUN_DIR if printed."""
    proc = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    run_dir = None
    for line in proc.stdout:
        print(line, end='')
        if line.startswith('RUN_DIR='):
            run_dir = line.strip().split('=', 1)[1]
    if proc.wait() != 0:
        raise subprocess.CalledProcessError(proc.returncode, args)
    return run_dir

def cli(*args):
    return run(sys.executable, '-m', 'steam_nffmtl.cli', *args)

def run_dirs(*args):
    """For multi-seed `train`: return every RUN_DIR line printed by the command."""
    out = subprocess.run([sys.executable, '-m', 'steam_nffmtl.cli', *map(str, args)], capture_output=True, text=True)
    print(out.stdout[-4000:], out.stderr[-4000:])
    if out.returncode != 0:
        raise subprocess.CalledProcessError(out.returncode, args)
    return [l.split('=', 1)[1] for l in out.stdout.splitlines() if l.startswith('RUN_DIR=')]

In [ ]:
from pathlib import Path

klasor = Path("/content/drive/MyDrive/steamRS")
print("steamRS klasörü:", klasor.is_dir())

for ad in ("steam-nffmtl-v0.3.1.zip", "recommendations.csv", "games_metadata.json"):
    print(ad, "→", "VAR" if (klasor / ad).is_file() else "YOK")

steamRS klasörü: False
steam-nffmtl-v0.3.1.zip → YOK
recommendations.csv → YOK
games_metadata.json → YOK


## 0. Setup (A100, High-RAM runtime)
Upload `steam-nffmtl-v0.3.1.zip` to `MyDrive/steamRS/` next to `recommendations.csv` and `games_metadata.json`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = Path('/content/drive/MyDrive/steamRS')
run('unzip', '-q', '-o', DATA_DIR / 'steam-nffmtl-v0.3.1.zip', '-d', '/content/')
REPO = Path('/content/steam-nffmtl-v0.3.1')
%cd {REPO}
run(sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt')
run(sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[test]')
sys.path.insert(0, str(REPO / 'src'))

Mounted at /content/drive
/content/steam-nffmtl-v0.3.1


In [ ]:
cli('preflight', '--require-cuda', '--out', DATA_DIR / 'preflight')

/usr/local/lib/python3.13/dist-packages/implicit/gpu/__init__.py:28: UserWarning: Disabling GPU support because of '/usr/local/lib/python3.13/dist-packages/implicit/gpu/_cuda.so: undefined symbol: _ZN3rmm13device_bufferC1EmNS_16cuda_stream_viewENS_6detail23cccl_async_resource_refIN4cuda2mr3__412resource_refIJNS6_17device_accessibleEEEEEE'
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
{
  "ok": true,
  "checks": {
    "import numpy": true,
    "import pandas": true,
    "import scipy": true,
    "import torch": true,
    "import implicit": true,
    "import yaml": true,
    "implicit user x item orientation": true,
    "

## 1. Tests (GPU tests included; skipped/failed tests are listed by `-rs`). Stop if anything fails.

In [ ]:
run(sys.executable, '-m', 'pytest', '-q', '-rs')

...............................................................          [100%]
=============================== warnings summary ===============================
tests/test_baselines.py::test_implicit_orientation_and_scores[als]
  /usr/local/lib/python3.13/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
    check_blas_config()

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
63 passed, 1 warning in 38.54s


## 2. Prepare the data
Cache folder named by the preprocessing settings; size/mtime checked on every load, SHA-256 re-computed when they differ (always in `final`, `verify_source_hash: true`).

In [ ]:
prep = Path(cli('prepare', '--config', 'configs/default.yaml'))
card = json.loads((prep / 'data_card.json').read_text())
print(json.dumps({k: card[k] for k in ['raw', 'duplicates_removed', 'repeated_pairs_across_partitions', 'kcore', 'train', 'val', 'test', 'tags', 'unverified_assumptions']}, indent=1))
cli('estimate', '--config', 'configs/default.yaml', '--variant', 'full')

2026-11-09 09:33:33,665 INFO run directory: /content/drive/MyDrive/steamRS/runs_tuning/2026-11-09-093332-tuning-prepare
2026-11-09 09:38:16,303 INFO prepared in 283s: 1000980 users, 23336 games, 11478966 training rows; fingerprint b8ea53ea1552
RUN_DIR=/content/drive/MyDrive/steamRS/runs_tuning/2026-11-09-093332-tuning-prepare
{
 "raw": {
  "date_max": "2022-12-31",
  "date_min": "2010-10-15",
  "games_in_reviews": 37610,
  "metadata_bad_lines": 0,
  "metadata_games": 50872,
  "rows": 41154794,
  "rows_outside_split_years": 0,
  "users": 13781059
 },
 "duplicates_removed": {
  "test": 6,
  "train": 6,
  "val": 8
 },
 "repeated_pairs_across_partitions": {
  "test_pairs_also_in_train_or_val": 1,
  "val_pairs_also_in_train": 0
 },
 "kcore": {
  "applied_to": "training years only",
  "games_after": 23336,
  "games_before": 26532,
  "iterations": 4,
  "k": 5,
  "rows_after": 11478966,
  "rows_before": 22762404,
  "sparsity_after": 0.9995085821208893,
  "users_after": 1000980,
  "users_before

## 3a. Probe: limited training batches (first GPU run)
The batch limit shortens NFF-MTL training only. Data loading, the training tensors on the GPU and the validation
pass use the full prepared data; baseline fits in the next cell also use the full training data (with few BPR/ALS iterations).

In [ ]:
probe = Path(cli('train', '--config', 'configs/probe.yaml', '--variant', 'full', '--seeds', '0'))
h = json.loads((probe / 'training.json').read_text())['history'][0]
print({k: h[k] for k in ['batches', 'full_epoch_batches', 'seconds_per_batch', 'val_seconds', 'val_users', 'peak_gpu_memory_gb', 'host_max_rss_gb']})
print('extrapolated full epoch (min):', round(h['seconds_per_batch'] * h['full_epoch_batches'] / 60, 1))

2026-11-09 09:39:42,213 INFO run directory: /content/drive/MyDrive/steamRS/runs_probe/2026-11-09-093942-pilot-train-full-seed0
2026-11-09 09:40:02,113 INFO variant=full seed=0 pos_weight=4.949131586950451 rows=11478966 real/batch=819
2026-11-09 09:40:15,074 INFO epoch 1 | loss 1.2047 (bce 0.6463, l1 1.1167) | val ndcg@20 0.02596 | 11s + 2s
2026-11-09 09:40:22,778 INFO selected epoch 1; checkpoint /content/drive/MyDrive/steamRS/runs_probe/2026-11-09-093942-pilot-train-full-seed0/checkpoint_best.pt
RUN_DIR=/content/drive/MyDrive/steamRS/runs_probe/2026-11-09-093942-pilot-train-full-seed0
{'batches': 300, 'full_epoch_batches': 14016, 'seconds_per_batch': 0.03702, 'val_seconds': 1.8, 'val_users': 2000, 'peak_gpu_memory_gb': 3.375, 'host_max_rss_gb': 3.151}
extrapolated full epoch (min): 8.6


Baseline probe (optional; ItemKNN-CF and TagKNN fit on the full training data, BPR 5 / ALS 2 iterations).

In [ ]:
probe_base = Path(cli('baselines', '--config', 'configs/probe.yaml', '--split', 'val', '--seed', '0'))
for f in sorted((probe_base / 'results').glob('*__val-positive.json')):
    m = json.loads(f.read_text()); print(m['model'], 'fit s:', m.get('fit_seconds'), 'eval s:', m['coverage']['seconds'])

2026-11-09 09:40:48,336 INFO run directory: /content/drive/MyDrive/steamRS/runs_probe/2026-11-09-094046-pilot-baselines-val-seed0
2026-11-09 09:41:20,315 INFO Random      val-positive      ndcg@20=0.00008 recall@20=0.00013 users=2000 (1s)
2026-11-09 09:41:20,350 INFO Random      val-interaction   ndcg@20=0.00042 recall@20=0.00117 users=2000 (1s)
2026-11-09 09:41:20,350 INFO Random      review-label AUC=0.5141 on 20000 rows; fit 0s
2026-11-09 09:41:22,532 INFO Popularity  val-positive      ndcg@20=0.03087 recall@20=0.06448 users=2000 (0s)
2026-11-09 09:41:22,584 INFO Popularity  val-interaction   ndcg@20=0.02884 recall@20=0.06340 users=2000 (0s)
2026-11-09 09:41:22,585 INFO Popularity  review-label AUC=0.5653 on 20000 rows; fit 0s
2026-11-09 09:41:29,184 INFO TagKNN      val-positive      ndcg@20=0.00748 recall@20=0.01361 users=2000 (1s)
2026-11-09 09:41:29,222 INFO TagKNN      val-interaction   ndcg@20=0.00814 recall@20=0.01207 users=2000 (1s)
2026-11-09 09:41:29,222 INFO TagKNN      r

In [ ]:
# ---- Gate 1. Read the probe numbers above (seconds per batch, extrapolated epoch, peak GPU memory, host RSS,
# baseline fit/eval seconds). Only then set this to True and run the cells below.
PROCEED_TO_PILOT = True
if not PROCEED_TO_PILOT:
    raise RuntimeError('Stopped after the probe by design. Review the probe output, then set PROCEED_TO_PILOT = True.')

## 3b. Pilot: full data, 2 epochs (run only if the probe extrapolation is acceptable)

In [ ]:
pilot_train = Path(cli('train', '--config', 'configs/pilot.yaml', '--variant', 'full', '--seeds', '0'))
print([(e['epoch'], e['train_seconds'], e['val_seconds'], e['peak_gpu_memory_gb']) for e in json.loads((pilot_train / 'training.json').read_text())['history']])

2026-11-09 09:47:21,479 INFO run directory: /content/drive/MyDrive/steamRS/runs_pilot/2026-11-09-094721-pilot-train-full-seed0
2026-11-09 09:47:43,269 INFO variant=full seed=0 pos_weight=4.949131586950451 rows=11478966 real/batch=819
2026-11-09 09:56:04,034 INFO epoch 1 | loss 0.9641 (bce 0.5208, l1 0.8867) | val ndcg@20 0.02468 | 496s + 5s
2026-11-09 10:04:22,647 INFO epoch 2 | loss 0.8967 (bce 0.4806, l1 0.8322) | val ndcg@20 0.02851 | 493s + 5s
2026-11-09 10:04:37,075 INFO selected epoch 2; checkpoint /content/drive/MyDrive/steamRS/runs_pilot/2026-11-09-094721-pilot-train-full-seed0/checkpoint_best.pt
RUN_DIR=/content/drive/MyDrive/steamRS/runs_pilot/2026-11-09-094721-pilot-train-full-seed0
[(1, 496.1, 4.7, 3.375), (2, 493.3, 4.7, 3.375)]


**Decide here.** Tuning runs `tuning.n_trials` trials for every model (`budget_policy: equal`; a smaller grid stops
`tune` before any trial). Equal trial counts are not equal compute time: one NFF-MTL trial is a full training run.
Final cost ≈ 15 NFF-MTL trainings (3 variants × 5 seeds) plus test evaluations. Lower `tuning.n_trials` with `--set`
if needed; it stays equal across models and is recorded in `selection.json`. Only models that were actually run may
be reported (EASE is off by default; NCF and LightGCN are not implemented).

## 4. Tuning on the validation year (equal budget)

In [ ]:
# ---- Gate 2. Decide the tuning budget from the pilot (see the note above) before running this cell.
PROCEED_TO_TUNE = True
INCLUDE_EASE = False   # enable only if `estimate` shows that EASE's peak memory fits this runtime
if not PROCEED_TO_TUNE:
    raise RuntimeError('Stopped before tuning by design. Set PROCEED_TO_TUNE = True after reviewing the pilot.')
models = ['tagknn', 'itemknn', 'bpr', 'als', 'nffmtl'] + (['ease'] if INCLUDE_EASE else [])
tune_args = ['tune', '--config', 'configs/default.yaml', '--models', *models]
if INCLUDE_EASE:
    tune_args += ['--set', 'baselines.ease.enabled=true']
tune_args += ['--set', 'tuning.n_trials=3', 'train.max_epochs=5']
tune = Path(cli(*tune_args))
print((tune / 'selection.json').read_text())   # trials_used, grid_size, budget_policy per model
FINAL_CFG = tune / 'final_selected.yaml'       # stage final, selected hyperparameters, strict source hashing

2026-11-09 10:22:39,028 INFO run directory: /content/drive/MyDrive/steamRS/runs_tuning/2026-11-09-102238-tuning-tune
2026-11-09 10:23:11,646 INFO tune TagKNN     trial 1/3 ndcg@20=0.00631 {'k': 400}
2026-11-09 10:23:23,899 INFO tune TagKNN     trial 2/3 ndcg@20=0.00940 {'k': 25}
2026-11-09 10:23:36,723 INFO tune TagKNN     trial 3/3 ndcg@20=0.00724 {'k': 150}
2026-11-09 10:23:57,316 INFO tune ItemKNN-CF trial 1/3 ndcg@20=0.06006 {'k': 100, 'shrink': 100.0}
2026-11-09 10:24:17,892 INFO tune ItemKNN-CF trial 2/3 ndcg@20=0.05899 {'k': 100, 'shrink': 10.0}
2026-11-09 10:24:39,943 INFO tune ItemKNN-CF trial 3/3 ndcg@20=0.06134 {'k': 400, 'shrink': 50.0}
/usr/local/lib/python3.13/dist-packages/implicit/gpu/__init__.py:28: UserWarning: Disabling GPU support because of '/usr/local/lib/python3.13/dist-packages/implicit/gpu/_cuda.so: undefined symbol: _ZN3rmm13device_bufferC1EmNS_16cuda_stream_viewENS_6detail23cccl_async_resource_refIN4cuda2mr3__412resource_refIJNS6_17device_accessibleEEEEEE'
  

In [ ]:
val_check = Path(cli(
    'train', '--config', FINAL_CFG,
    '--variant', 'full', '--seeds', '0',
    '--set', 'experiment.stage=tuning', 'train.max_epochs=10',
))

2026-11-09 12:41:30,440 INFO run directory: /content/drive/MyDrive/steamRS/runs_final/2026-11-09-124130-tuning-train-full-seed0
2026-11-09 12:41:52,255 INFO variant=full seed=0 pos_weight=4.949131586950451 rows=11478966 real/batch=819
2026-11-09 12:49:24,586 INFO epoch 1 | loss 0.9513 (bce 0.5121, l1 0.8786) | val ndcg@20 0.02835 | 434s + 19s
2026-11-09 12:56:54,420 INFO epoch 2 | loss 0.8279 (bce 0.4380, l1 0.7799) | val ndcg@20 0.03080 | 431s + 19s
2026-11-09 13:04:25,836 INFO epoch 3 | loss 0.8069 (bce 0.4239, l1 0.7659) | val ndcg@20 0.03087 | 432s + 19s
2026-11-09 13:11:55,645 INFO epoch 4 | loss 0.7539 (bce 0.3933, l1 0.7212) | val ndcg@20 0.03310 | 431s + 19s
2026-11-09 13:19:25,967 INFO epoch 5 | loss 0.6946 (bce 0.3601, l1 0.6690) | val ndcg@20 0.03518 | 431s + 18s
2026-11-09 13:27:00,339 INFO epoch 6 | loss 0.6438 (bce 0.3317, l1 0.6242) | val ndcg@20 0.03533 | 435s + 19s
2026-11-09 13:34:30,673 INFO epoch 7 | loss 0.6956 (bce 0.3627, l1 0.6659) | val ndcg@20 0.03438 | 431s +

## 5. Final: retrain with frozen settings, evaluate each checkpoint once on 2022, compare

In [ ]:
FINAL_CFG = Path("/content/drive/MyDrive/steamRS/runs_tuning/2026-11-09-102238-tuning-tune/final_selected.yaml")
final_full_seed0 = Path(cli(
    'train', '--config', FINAL_CFG,
    '--variant', 'full', '--seeds', '0',
    '--set', 'train.max_epochs=10',
))

2026-11-10 09:59:55,475 INFO run directory: /content/drive/MyDrive/steamRS/runs_final/2026-11-10-095955-final-train-full-seed0
2026-11-10 10:00:21,829 INFO variant=full seed=0 pos_weight=4.949131586950451 rows=11478966 real/batch=819
2026-11-10 10:07:53,746 INFO epoch 1 | loss 0.9513 (bce 0.5121, l1 0.8786) | val ndcg@20 0.02835 | 433s + 19s
2026-11-10 10:15:26,301 INFO epoch 2 | loss 0.8279 (bce 0.4380, l1 0.7799) | val ndcg@20 0.03080 | 433s + 19s
2026-11-10 10:23:01,126 INFO epoch 3 | loss 0.8069 (bce 0.4239, l1 0.7659) | val ndcg@20 0.03087 | 435s + 19s
2026-11-10 10:30:33,400 INFO epoch 4 | loss 0.7539 (bce 0.3933, l1 0.7212) | val ndcg@20 0.03310 | 433s + 19s
2026-11-10 10:38:05,997 INFO epoch 5 | loss 0.6946 (bce 0.3601, l1 0.6690) | val ndcg@20 0.03518 | 433s + 19s
2026-11-10 10:45:36,950 INFO epoch 6 | loss 0.6438 (bce 0.3317, l1 0.6242) | val ndcg@20 0.03533 | 432s + 19s
2026-11-10 10:53:11,654 INFO epoch 7 | loss 0.6956 (bce 0.3627, l1 0.6659) | val ndcg@20 0.03438 | 435s + 

In [ ]:
final_eval_full_seed0 = Path(cli(
    'evaluate', '--config', FINAL_CFG,
    '--split', 'test', '--run', final_full_seed0,
    '--set', 'train.max_epochs=10',
))

2026-11-10 11:24:26,910 INFO run directory: /content/drive/MyDrive/steamRS/runs_final/2026-11-10-112426-final-evaluate-test-full-seed0
2026-11-10 11:35:26,229 INFO NFF-MTL test-positive ndcg@20=0.03261 recall@20=0.07192
2026-11-10 11:35:31,500 INFO NFF-MTL test-interaction ndcg@20=0.03000 recall@20=0.06489
2026-11-10 11:35:49,443 INFO playtime log-MAE: {'NFF-MTL': 0.8907, 'GlobalMean': 1.2425, 'GameMedian': 0.9101, 'UserGameMean': 0.9116}
RUN_DIR=/content/drive/MyDrive/steamRS/runs_final/2026-11-10-112426-final-evaluate-test-full-seed0


In [ ]:
final_baselines_seed0 = Path(cli(
    'baselines', '--config', FINAL_CFG,
    '--split', 'test', '--seed', '0',
    '--set', 'train.max_epochs=10',
))

2026-11-10 11:40:04,034 INFO run directory: /content/drive/MyDrive/steamRS/runs_final/2026-11-10-114002-final-baselines-test-seed0
2026-11-10 11:49:06,675 INFO Random      test-positive     ndcg@20=0.00037 recall@20=0.00084 users=304199 (177s)
2026-11-10 11:49:11,216 INFO Random      test-interaction  ndcg@20=0.00038 recall@20=0.00085 users=345099 (231s)
2026-11-10 11:49:11,216 INFO Random      review-label AUC=0.4988 on 500000 rows; fit 0s
2026-11-10 11:53:31,196 INFO Popularity  test-positive     ndcg@20=0.02797 recall@20=0.06140 users=304199 (100s)
2026-11-10 11:53:36,430 INFO Popularity  test-interaction  ndcg@20=0.02819 recall@20=0.06058 users=345099 (114s)
2026-11-10 11:53:36,431 INFO Popularity  review-label AUC=0.5602 on 500000 rows; fit 0s
2026-11-10 11:58:59,869 INFO TagKNN      test-positive     ndcg@20=0.00663 recall@20=0.01232 users=304199 (99s)
2026-11-10 11:59:04,687 INFO TagKNN      test-interaction  ndcg@20=0.00647 recall@20=0.01183 users=345099 (122s)
2026-11-10 11:59

In [ ]:
first_compare = Path(cli(
    'compare', '--config', FINAL_CFG,
    '--split', 'test-positive', '--reference', 'NFF-MTL',
    '--results', final_eval_full_seed0, final_baselines_seed0,
    '--metrics', 'ndcg@20', 'recall@20',
))

2026-11-10 13:14:14,090 INFO run directory: /content/drive/MyDrive/steamRS/runs_final/2026-11-10-131412-final-compare
2026-11-10 13:15:06,936 INFO 
            runs  ndcg@20_mean  recall@20_mean  arp@10_mean  arp@20_mean  coverage@10_mean  coverage@20_mean  gini@10_mean  gini@20_mean  long_tail_share@10_mean  long_tail_share@20_mean  novelty@10_mean  novelty@20_mean  tag_ild@10_mean  tag_ild@20_mean  auc_review_label_mean
model                                                                                                                                                                                                                                                                                
ALS            1       0.05268         0.10561  20463.55663  19113.71155           0.03013           0.03681       0.98931       0.98750                  0.00000                  0.00000          9.37035          9.49574          0.47866          0.50984                0.55628
BPR-MF         1  

In [ ]:
final_full_seed1 = Path(cli(
    'train', '--config', FINAL_CFG,
    '--variant', 'full', '--seeds', '1',
    '--set', 'train.max_epochs=10',
))

2026-11-10 13:43:58,715 INFO run directory: /content/drive/MyDrive/steamRS/runs_final/2026-11-10-134358-final-train-full-seed1
2026-11-10 13:44:20,670 INFO variant=full seed=1 pos_weight=4.949131586950451 rows=11478966 real/batch=819
2026-11-10 13:51:59,281 INFO epoch 1 | loss 0.9513 (bce 0.5119, l1 0.8789) | val ndcg@20 0.02995 | 440s + 19s
2026-11-10 13:59:36,946 INFO epoch 2 | loss 0.8271 (bce 0.4375, l1 0.7792) | val ndcg@20 0.03232 | 438s + 19s
2026-11-10 14:07:11,674 INFO epoch 3 | loss 0.8050 (bce 0.4221, l1 0.7658) | val ndcg@20 0.03159 | 435s + 19s
2026-11-10 14:14:45,868 INFO epoch 4 | loss 0.7516 (bce 0.3908, l1 0.7216) | val ndcg@20 0.03324 | 436s + 19s
2026-11-10 14:22:20,444 INFO epoch 5 | loss 0.6925 (bce 0.3576, l1 0.6697) | val ndcg@20 0.03512 | 435s + 19s
2026-11-10 14:29:56,186 INFO epoch 6 | loss 0.6419 (bce 0.3295, l1 0.6248) | val ndcg@20 0.03551 | 436s + 19s
2026-11-10 14:37:32,327 INFO epoch 7 | loss 0.6933 (bce 0.3601, l1 0.6664) | val ndcg@20 0.03553 | 437s + 

In [ ]:
from pathlib import Path

FINAL_CFG = Path(
    "/content/drive/MyDrive/steamRS/runs_tuning/"
    "2026-11-09-102238-tuning-tune/final_selected.yaml"
)

final_full_seed1 = Path(
    "/content/drive/MyDrive/steamRS/runs_final/"
    "2026-11-10-134358-final-train-full-seed1"
)

final_eval_full_seed1 = Path(cli(
    'evaluate', '--config', FINAL_CFG,
    '--split', 'test', '--run', final_full_seed1,
    '--set', 'train.max_epochs=10',
))

2026-11-12 11:05:57,727 INFO run directory: /content/drive/MyDrive/steamRS/runs_final/2026-11-12-110556-final-evaluate-test-full-seed1
2026-11-12 11:17:04,849 INFO NFF-MTL test-positive ndcg@20=0.03284 recall@20=0.07287
2026-11-12 11:17:10,212 INFO NFF-MTL test-interaction ndcg@20=0.03042 recall@20=0.06615
2026-11-12 11:17:28,686 INFO playtime log-MAE: {'NFF-MTL': 0.8935, 'GlobalMean': 1.2425, 'GameMedian': 0.9101, 'UserGameMean': 0.9116}
RUN_DIR=/content/drive/MyDrive/steamRS/runs_final/2026-11-12-110556-final-evaluate-test-full-seed1


In [ ]:
from pathlib import Path

FINAL_CFG = Path(
    "/content/drive/MyDrive/steamRS/runs_tuning/"
    "2026-11-09-102238-tuning-tune/final_selected.yaml"
)
print("Ayar dosyası bulundu:", FINAL_CFG.exists())

Ayar dosyası bulundu: True


In [ ]:
final_full_seed2 = Path(cli(
    'train', '--config', FINAL_CFG,
    '--variant', 'full', '--seeds', '2',
    '--set', 'train.max_epochs=10',
))

2026-11-14 15:11:06,631 INFO run directory: /content/drive/MyDrive/steamRS/runs_final/2026-11-14-151106-final-train-full-seed2
2026-11-14 15:11:32,467 INFO variant=full seed=2 pos_weight=4.949131586950451 rows=11478966 real/batch=819
2026-11-14 15:19:05,355 INFO epoch 1 | loss 0.9519 (bce 0.5130, l1 0.8778) | val ndcg@20 0.02829 | 434s + 19s
2026-11-14 15:26:43,016 INFO epoch 2 | loss 0.8286 (bce 0.4390, l1 0.7791) | val ndcg@20 0.03141 | 438s + 19s
2026-11-14 15:34:19,391 INFO epoch 3 | loss 0.8063 (bce 0.4240, l1 0.7646) | val ndcg@20 0.03532 | 437s + 19s


In [ ]:
d = DATA_DIR / 'runs_final' / '2026-11-12-132154-final-train-full-seed2'
print('training.json var mı:', (d / 'training.json').exists())
print((d / 'run.log').read_text()[-1500:])

training.json var mı: True
2026-11-12 13:21:54,604 INFO run directory: /content/drive/MyDrive/steamRS/runs_final/2026-11-12-132154-final-train-full-seed2
2026-11-12 13:22:20,227 INFO variant=full seed=2 pos_weight=4.949131586950451 rows=11478966 real/batch=819
2026-11-12 13:29:56,399 INFO epoch 1 | loss 0.9519 (bce 0.5130, l1 0.8778) | val ndcg@20 0.02829 | 438s + 19s
2026-11-12 13:37:31,838 INFO epoch 2 | loss 0.8286 (bce 0.4390, l1 0.7791) | val ndcg@20 0.03141 | 436s + 19s
2026-11-12 13:45:08,959 INFO epoch 3 | loss 0.8063 (bce 0.4240, l1 0.7646) | val ndcg@20 0.03532 | 438s + 19s
2026-11-12 13:52:46,077 INFO epoch 4 | loss 0.7527 (bce 0.3927, l1 0.7201) | val ndcg@20 0.03529 | 438s + 19s
2026-11-12 14:00:19,038 INFO epoch 5 | loss 0.6939 (bce 0.3596, l1 0.6685) | val ndcg@20 0.03664 | 434s + 19s
2026-11-12 14:07:52,649 INFO epoch 6 | loss 0.6434 (bce 0.3313, l1 0.6241) | val ndcg@20 0.03588 | 434s + 19s
2026-11-12 14:15:25,527 INFO epoch 7 | loss 0.6952 (bce 0.3624, l1 0.6657) | va

In [ ]:
print([p.name for p in (DATA_DIR / 'runs_final').glob('*final-train-full-seed2*')])

['2026-11-12-132154-final-train-full-seed2', '2026-11-12-144140-final-train-full-seed2']


In [ ]:
import subprocess, time
subprocess.run(['pkill', '-f', 'steam_nffmtl.cli train'])
time.sleep(5)
print(subprocess.run(['bash', '-c', 'ps -eo pid,etime,cmd | grep "steam_nffmtl.cli train" | grep -v grep'], capture_output=True, text=True).stdout or 'çalışan eğitim yok')
dup = DATA_DIR / 'runs_final' / '2026-11-12-144140-final-train-full-seed2'
dup.rename(dup.with_name(dup.name + '-INTERRUPTED'))

çalışan eğitim yok


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/steamRS/runs_final/2026-11-12-144140-final-train-full-seed2' -> '/content/drive/MyDrive/steamRS/runs_final/2026-11-12-144140-final-train-full-seed2-INTERRUPTED'

In [ ]:
final_full_seed2 = DATA_DIR / 'runs_final' / '2026-11-12-132154-final-train-full-seed2'
final_eval_full_seed2 = Path(cli(
    'evaluate', '--config', FINAL_CFG,
    '--split', 'test', '--run', final_full_seed2,
    '--set', 'train.max_epochs=10',
))

2026-11-12 14:47:48,521 INFO run directory: /content/drive/MyDrive/steamRS/runs_final/2026-11-12-144748-final-evaluate-test-full-seed2
2026-11-12 14:58:44,747 INFO NFF-MTL test-positive ndcg@20=0.03513 recall@20=0.07629
2026-11-12 14:58:50,078 INFO NFF-MTL test-interaction ndcg@20=0.03221 recall@20=0.06855
2026-11-12 14:59:07,983 INFO playtime log-MAE: {'NFF-MTL': 0.8913, 'GlobalMean': 1.2425, 'GameMedian': 0.9101, 'UserGameMean': 0.9116}
RUN_DIR=/content/drive/MyDrive/steamRS/runs_final/2026-11-12-144748-final-evaluate-test-full-seed2


In [ ]:
final_full_seed3 = Path(cli(
    'train', '--config', FINAL_CFG,
    '--variant', 'full', '--seeds', '3',
    '--set', 'train.max_epochs=10',
))

2026-11-13 09:46:55,654 INFO run directory: /content/drive/MyDrive/steamRS/runs_final/2026-11-13-094655-final-train-full-seed3
2026-11-13 09:47:21,232 INFO variant=full seed=3 pos_weight=4.949131586950451 rows=11478966 real/batch=819
2026-11-13 09:54:53,286 INFO epoch 1 | loss 0.9517 (bce 0.5127, l1 0.8780) | val ndcg@20 0.02909 | 433s + 19s
2026-11-13 10:02:24,572 INFO epoch 2 | loss 0.8280 (bce 0.4376, l1 0.7809) | val ndcg@20 0.03035 | 432s + 19s
2026-11-13 10:09:56,241 INFO epoch 3 | loss 0.8069 (bce 0.4229, l1 0.7679) | val ndcg@20 0.03325 | 432s + 19s
2026-11-13 10:17:27,982 INFO epoch 4 | loss 0.7553 (bce 0.3929, l1 0.7248) | val ndcg@20 0.03441 | 432s + 19s
2026-11-13 10:24:59,392 INFO epoch 5 | loss 0.6975 (bce 0.3604, l1 0.6742) | val ndcg@20 0.03542 | 432s + 19s
2026-11-13 10:32:30,802 INFO epoch 6 | loss 0.6474 (bce 0.3321, l1 0.6305) | val ndcg@20 0.03501 | 432s + 19s
2026-11-13 10:40:01,540 INFO epoch 7 | loss 0.7001 (bce 0.3642, l1 0.6718) | val ndcg@20 0.03383 | 432s + 

In [ ]:
final_eval_full_seed3 = Path(cli(
    'evaluate', '--config', FINAL_CFG,
    '--split', 'test', '--run', final_full_seed3,
    '--set', 'train.max_epochs=10',
))

2026-11-13 12:19:46,202 INFO run directory: /content/drive/MyDrive/steamRS/runs_final/2026-11-13-121946-final-evaluate-test-full-seed3
2026-11-13 12:30:39,501 INFO NFF-MTL test-positive ndcg@20=0.03252 recall@20=0.07300
2026-11-13 12:30:44,732 INFO NFF-MTL test-interaction ndcg@20=0.02986 recall@20=0.06592
2026-11-13 12:31:02,536 INFO playtime log-MAE: {'NFF-MTL': 0.8836, 'GlobalMean': 1.2425, 'GameMedian': 0.9101, 'UserGameMean': 0.9116}
RUN_DIR=/content/drive/MyDrive/steamRS/runs_final/2026-11-13-121946-final-evaluate-test-full-seed3


In [ ]:
final_full_seed4 = Path(cli('train', '--config', FINAL_CFG, '--variant', 'full', '--seeds', '4', '--set', 'train.max_epochs=10'))
final_eval_full_seed4 = Path(cli('evaluate', '--config', FINAL_CFG, '--split', 'test', '--run', final_full_seed4, '--set', 'train.max_epochs=10'))

2026-11-13 12:35:10,928 INFO run directory: /content/drive/MyDrive/steamRS/runs_final/2026-11-13-123510-final-train-full-seed4
2026-11-13 12:35:31,709 INFO variant=full seed=4 pos_weight=4.949131586950451 rows=11478966 real/batch=819
2026-11-13 12:43:01,877 INFO epoch 1 | loss 0.9526 (bce 0.5136, l1 0.8780) | val ndcg@20 0.02859 | 431s + 19s
2026-11-13 12:50:31,843 INFO epoch 2 | loss 0.8300 (bce 0.4402, l1 0.7795) | val ndcg@20 0.03162 | 431s + 19s
2026-11-13 12:58:01,747 INFO epoch 3 | loss 0.8077 (bce 0.4253, l1 0.7647) | val ndcg@20 0.03326 | 431s + 19s
2026-11-13 13:05:31,899 INFO epoch 4 | loss 0.7545 (bce 0.3948, l1 0.7195) | val ndcg@20 0.03508 | 431s + 19s
2026-11-13 13:13:02,108 INFO epoch 5 | loss 0.6953 (bce 0.3616, l1 0.6673) | val ndcg@20 0.03558 | 431s + 19s
2026-11-13 13:20:31,918 INFO epoch 6 | loss 0.6449 (bce 0.3335, l1 0.6227) | val ndcg@20 0.03664 | 431s + 19s
2026-11-13 13:28:02,897 INFO epoch 7 | loss 0.6960 (bce 0.3638, l1 0.6644) | val ndcg@20 0.03584 | 432s + 

In [ ]:
s = 2  # sonra 1, 2, 3, 4
tr = Path(cli('train', '--config', FINAL_CFG, '--variant', 'no_content', '--seeds', s, '--set', 'train.max_epochs=10'))
ev = Path(cli('evaluate', '--config', FINAL_CFG, '--split', 'test', '--run', tr, '--set', 'train.max_epochs=10'))
print('BİTTİ', s, tr, ev)

2026-11-15 11:59:39,701 INFO run directory: /content/drive/MyDrive/steamRS/runs_final/2026-11-15-115939-final-train-no_content-seed2
2026-11-15 11:59:59,669 INFO variant=no_content seed=2 pos_weight=4.949131586950451 rows=11478966 real/batch=819
2026-11-15 12:07:25,800 INFO epoch 1 | loss 0.9621 (bce 0.5206, l1 0.8830) | val ndcg@20 0.02630 | 428s + 18s
2026-11-15 12:14:52,229 INFO epoch 2 | loss 0.8534 (bce 0.4612, l1 0.7843) | val ndcg@20 0.02788 | 427s + 19s
2026-11-15 12:22:18,975 INFO epoch 3 | loss 0.8309 (bce 0.4455, l1 0.7709) | val ndcg@20 0.03252 | 427s + 19s
2026-11-15 12:29:45,655 INFO epoch 4 | loss 0.7773 (bce 0.4140, l1 0.7266) | val ndcg@20 0.03154 | 427s + 19s
2026-11-15 12:37:11,298 INFO epoch 5 | loss 0.7183 (bce 0.3807, l1 0.6752) | val ndcg@20 0.03347 | 427s + 18s
2026-11-15 12:44:37,951 INFO epoch 6 | loss 0.6673 (bce 0.3520, l1 0.6307) | val ndcg@20 0.03266 | 427s + 19s
2026-11-15 12:52:03,535 INFO epoch 7 | loss 0.7174 (bce 0.3815, l1 0.6719) | val ndcg@20 0.032

In [ ]:
final_runs = []
# final_runs.append(cli('baselines', '--config', FINAL_CFG, '--split', 'test', '--seed', 0))
# for s in [1, 2, 3, 4]:   # seed-dependent baselines only
#     final_runs.append(cli('baselines', '--config', FINAL_CFG, '--split', 'test', '--seed', s, '--only', 'Random', 'BPR-MF', 'ALS'))
# for variant in ['full', 'no_content', 'single_task']:
#     for train_dir in run_dirs('train', '--config', FINAL_CFG, '--variant', variant, '--seeds', 0, 1, 2, 3, 4):
#         final_runs.append(cli('evaluate', '--config', FINAL_CFG, '--split', 'test', '--run', train_dir))
# cmp = Path(cli('compare', '--config', FINAL_CFG, '--reference', 'NFF-MTL', '--results', *final_runs))
# import pandas as pd; pd.read_csv(cmp / 'comparison_table.csv', index_col=0).filter(regex='_mean$|runs').round(4)